SETUP

In [10]:
import tensorflow as tf
import cv2
import numpy as np
import time

MODEL_PATH = "hrnet_pose.tflite"
DELEGATE_PATH = "libQnnTFLiteDelegate.so"

def load_interpreter(model_path, use_npu=True):
    if use_npu:
        try:
            delegate = tf.lite.experimental.load_delegate(DELEGATE_PATH, options={"backend_type": "htp"})
            interp = tf.lite.Interpreter(model_path=model_path, experimental_delegates=[delegate])
            interp.allocate_tensors()
            print("Loaded QNN HTP (NPU) delegate.")
            return interp, "npu"
        except (ValueError, OSError) as e:
            print(f"Could not load NPU delegate ({e}). Falling back to CPU.")

    interp = tf.lite.Interpreter(model_path=model_path)
    interp.allocate_tensors()
    return interp, "cpu"

interpreter_npu, backend = load_interpreter("hrnet_pose.tflite", use_npu=True)
input_details = interpreter_npu.get_input_details()
output_details = interpreter_npu.get_output_details()
scale, zero_point = output_details[0]["quantization"]

IN_H, IN_W = input_details[0]["shape"][1], input_details[0]["shape"][2]

print(f"Active backend: {backend}")
print(f"Input size (H,W): ({IN_H}, {IN_W})")
print(f"Output quantization: scale={scale}, zero_point={zero_point}")


Starting stage: Graph Preparation Initializing
Completed stage: Graph Preparation Initializing (305 us)
Starting stage: Graph Optimizations
Loaded QNN HTP (NPU) delegate.
Completed stage: Graph Optimizations (1520505 us)
Starting stage: Post Graph Optimization
Completed stage: Post Graph Optimization (49823 us)
Starting stage: Graph Sequencing for Target
Completed stage: Graph Sequencing for Target (174789 us)
Starting stage: VTCM Allocation
Completed stage: VTCM Allocation (24038 us)
Starting stage: Parallelization Optimization
Completed stage: Parallelization Optimization (27360 us)
Starting stage: Finalizing Graph Sequence

====== DDR bandwidth summary ======
spill_bytes=0
fill_bytes=0
write_total_bytes=98304
read_total_bytes=29335552

Completed stage: Finalizing Graph Sequence (23672 us)
Starting stage: Completion
Completed stage: Completion (2021 us)
Active backend: npu
Input size (H,W): (256, 192)
Output quantization: scale=0.004067708272486925, zero_point=8


/home/ubuntu/qai/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [11]:
COCO_KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle",
]

def get_joint_xy_conf(name, heatmaps, img_w, img_h, hm_w, hm_h):
    j = COCO_KEYPOINT_NAMES.index(name)
    jh = heatmaps[:, :, j].astype(np.float64)
    y, x = np.unravel_index(np.argmax(jh), jh.shape)
    conf = (float(jh[y, x]) - zero_point) * scale

    # Sub-pixel refinement via parabolic interpolation around the peak bin.
    # Raw argmax only has heatmap-grid precision (can be tens of pixels on a
    # high-res source frame), which was causing large spurious frame-to-frame
    # jitter in downstream angle calculations even when confidence was high.
    def _refine(center, lo, hi):
        denom = lo - 2 * center + hi
        if abs(denom) < 1e-6:
            return 0.0
        return 0.5 * (lo - hi) / denom

    dx = dy = 0.0
    if 0 < x < jh.shape[1] - 1:
        dx = _refine(jh[y, x], jh[y, x - 1], jh[y, x + 1])
    if 0 < y < jh.shape[0] - 1:
        dy = _refine(jh[y, x], jh[y - 1, x], jh[y + 1, x])

    px = (x + dx) * img_w / hm_w
    py = (y + dy) * img_h / hm_h
    return (px, py), conf

def get_joint_xy_conf_filtered(name, heatmaps, img_w, img_h, hm_w, hm_h, prev_xy=None, dt=None,
                                 max_speed_px_per_sec=4000):
    """Same as get_joint_xy_conf, but rejects a detection that implies an
    impossible jump from the previous frame (fast motion blur can make the
    model confidently lock onto background instead of the joint). Falls back
    to the raw detection with confidence zeroed out, so downstream code
    treats it as low-confidence rather than trusting a bad position."""
    xy, conf = get_joint_xy_conf(name, heatmaps, img_w, img_h, hm_w, hm_h)
    if prev_xy is not None and dt is not None and dt > 0:
        speed = np.linalg.norm(np.array(xy) - np.array(prev_xy)) / dt
        if speed > max_speed_px_per_sec:
            return xy, 0.0  # flag as untrustworthy; position kept for reference only
    return xy, conf

def joint_angle(a, b, c):
    """Angle at point b, formed by points a-b-c, in degrees."""
    a, b, c = np.array(a, dtype=np.float64), np.array(b, dtype=np.float64), np.array(c, dtype=np.float64)
    ba = a - b
    bc = c - b
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))

In [12]:
def extract_pose_signals(video_path):
    """Runs the full clip through the model once, returns per-frame joint positions + confidences."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    tracked_joints = [
        "nose",
        "left_shoulder", "right_shoulder",
        "left_elbow", "right_elbow",
        "left_wrist", "right_wrist",
        "left_hip", "right_hip",
        "left_knee", "right_knee",
        "left_ankle", "right_ankle",
    ]

    data = {"timestamps": []}
    for name in tracked_joints:
        data[f"{name}_xy"] = []
        data[f"{name}_conf"] = []

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (IN_W, IN_H))
        model_input = np.expand_dims(resized, axis=0).astype(np.uint8)

        interpreter_npu.set_tensor(input_details[0]["index"], model_input)
        interpreter_npu.invoke()
        heatmaps = interpreter_npu.get_tensor(output_details[0]["index"])[0]

        hm_h, hm_w, _ = heatmaps.shape
        img_h, img_w = frame.shape[0], frame.shape[1]

        for name in tracked_joints:
            xy, conf = get_joint_xy_conf(name, heatmaps, img_w, img_h, hm_w, hm_h)
            data[f"{name}_xy"].append(xy)
            data[f"{name}_conf"].append(conf)

        data["timestamps"].append(frame_idx / fps)
        frame_idx += 1

    cap.release()
    for key in data:
        data[key] = np.array(data[key])
    data["fps"] = fps
    return data


In [13]:
def smooth(signal, window=3):
    if len(signal) < window:
        return signal
    return np.convolve(signal, np.ones(window)/window, mode="same")


def compute_onset_by_velocity(values, timestamps, confidences, confidence_threshold=0.4,
                                velocity_threshold_multiplier=3.0, smoothing_window=3):
    valid = confidences >= confidence_threshold
    if valid.sum() < 5:
        return None

    t = timestamps[valid]
    v = smooth(values[valid], smoothing_window)
    velocity = np.abs(np.diff(v)) / np.diff(t)
    v_t = t[1:]

    n_ref = max(3, int(0.15 * len(velocity)))
    resting_mean = velocity[:n_ref].mean()
    resting_std = velocity[:n_ref].std() + 1e-6
    threshold = resting_mean + velocity_threshold_multiplier * resting_std

    for ti, vi in zip(v_t[n_ref:], velocity[n_ref:]):
        if vi > threshold:
            return ti
    return None

'''def compute_onset_by_angle_change(angles, timestamps, confidences, angle_change_threshold=15.0,
                                    confidence_threshold=0.5, reference_frames=5):
    valid = confidences >= confidence_threshold
    if valid.sum() < reference_frames + 1:
        return None

    t = timestamps[valid]
    a = angles[valid]

    reference_angle = np.median(a[:reference_frames])  # "resting" position, from a few real early frames

    for ti, ai in zip(t, a):
        if abs(ai - reference_angle) > angle_change_threshold:
            return ti
    return None'''


In [14]:
def sequencing_score(data):
    t = data["timestamps"]

    hip_angle = np.array([
        joint_angle(data["left_hip_xy"][i], data["right_hip_xy"][i], data["right_knee_xy"][i])
        for i in range(len(t))
    ])
    hip_conf = np.minimum(np.minimum(data["left_hip_conf"], data["right_hip_conf"]), data["right_knee_conf"])

    shoulder_angle = np.array([
        joint_angle(data["left_shoulder_xy"][i], data["right_shoulder_xy"][i], data["right_elbow_xy"][i])
        for i in range(len(t))
    ])
    shoulder_conf = np.minimum(np.minimum(data["left_shoulder_conf"], data["right_shoulder_conf"]), data["right_elbow_conf"])

    hip_onset = compute_onset_by_velocity(hip_angle, t, hip_conf)
    shoulder_onset = compute_onset_by_velocity(shoulder_angle, t, shoulder_conf)

    if hip_onset is None or shoulder_onset is None:
        return {"metric": "sequencing", "message": "Not enough confident data to score.", "verdict": None}

    gap = shoulder_onset - hip_onset
    verdict = gap > 0.02
    return {
        "metric": "sequencing",
        "hip_onset": hip_onset,
        "shoulder_onset": shoulder_onset,
        "gap_seconds": gap,
        "verdict": verdict,
        "message": "Great sequencing \u2014 hips rotating before shoulders." if verdict
                   else "Throw is becoming arm-dominant \u2014 shoulders rotating too early relative to hips.",
    }


In [16]:
def run_all_metrics(video_path, lead_leg="left"):
    data = extract_pose_signals(video_path)
    results = {
        "sequencing": sequencing_score(data),
    }
    for name, r in results.items():
        print(f"--- {name} ---")
        for k, v in r.items():
            print(f"  {k}: {v}")
        print()
    return results, data


results, data = run_all_metrics("1.mp4", lead_leg="left")


--- sequencing ---
  metric: sequencing
  hip_onset: 2.1704404958677688
  shoulder_onset: 2.4550884297520663
  gap_seconds: 0.28464793388429754
  verdict: True
  message: Great sequencing — hips rotating before shoulders.



LIVE CAMERA

In [18]:
import os
from collections import deque
import time

# --- tunables ---
PREROLL_SECONDS = 1.5       # widened from 1.0 — gives compute_onset_by_velocity's n_ref
                            # baseline-skip more real resting data, so an early onset (e.g.
                            # shoulders firing right away in an arm-dominant throw) is less
                            # likely to fall inside the skipped region
POST_TRIGGER_SECONDS = 3.5  # widened from 3.0 — keeps a true onset comfortably away from
                            # the tail edge, where the smoothing convolution's boundary
                            # distortion can produce a spurious velocity spike
RESULT_DISPLAY_SECONDS = 5.0
HIP_CONF_THRESHOLD = 0.4
VELOCITY_BASELINE_SAMPLES = 15   # frames of resting velocity to build the trigger threshold from
VELOCITY_MULTIPLIER = 4.0
MIN_TRIGGER_VELOCITY = 20.0      # deg/sec floor, so tiny jitter can't trigger even if it beats the baseline

SKEL_OUT_DIR = "./skeleton_output_stage1"
os.makedirs(SKEL_OUT_DIR, exist_ok=True)

tracked_joints = [
    "nose",
    "left_shoulder", "right_shoulder",
    "left_elbow", "right_elbow",
    "left_wrist", "right_wrist",
    "left_hip", "right_hip",
    "left_knee", "right_knee",
    "left_ankle", "right_ankle",
]

# Only edges between joints we actually track (a subset of full COCO, since
# eyes/ears aren't tracked here).
LIVE_SKELETON_EDGES = [
    ("left_shoulder", "left_elbow"),  ("left_elbow", "left_wrist"),
    ("right_shoulder", "right_elbow"), ("right_elbow", "right_wrist"),
    ("left_shoulder", "right_shoulder"),
    ("left_shoulder", "left_hip"),    ("right_shoulder", "right_hip"),
    ("left_hip", "right_hip"),
    ("left_hip", "left_knee"),        ("left_knee", "left_ankle"),
    ("right_hip", "right_knee"),      ("right_knee", "right_ankle"),
]
SKELETON_DRAW_CONF_THRESHOLD = 0.4

def get_frame_joint_data(heatmaps, img_w, img_h, hm_w, hm_h):
    """All tracked joints for one frame, in the same shape the metric functions expect."""
    out = {}
    for name in tracked_joints:
        xy, conf = get_joint_xy_conf(name, heatmaps, img_w, img_h, hm_w, hm_h)
        out[f"{name}_xy"] = xy
        out[f"{name}_conf"] = conf
    return out

def buffer_to_data_dict(frames, fps_estimate):
    """Converts a list of per-frame dicts (from the live loop) into the same
    `data` structure extract_pose_signals() produces, so the existing metric
    functions work unchanged."""
    data = {"timestamps": np.array([f["t"] for f in frames])}
    for name in tracked_joints:
        data[f"{name}_xy"] = np.array([f[f"{name}_xy"] for f in frames])
        data[f"{name}_conf"] = np.array([f[f"{name}_conf"] for f in frames])
    data["fps"] = fps_estimate
    return data

def sequencing_score(data):
    """Identical to test_vdo.ipynb's sequencing_score — both hip_onset and
    shoulder_onset are independently re-detected from the clip via
    compute_onset_by_velocity, matching the reference notebook exactly.

    (A previous version of this cell took a shortcut: using the live
    trigger's firing time directly as hip_onset instead of re-detecting it,
    to dodge a boundary-artifact bug. That shortcut introduced a worse
    problem: the trigger fires on HIP velocity, which is weak/late in an
    arm-dominant throw — so hip_onset ended up anchored late, while the
    real, early shoulder onset could fall inside shoulder_onset's own
    skipped 'resting baseline' window and get missed in favor of some later
    movement. That combination could make an arm-dominant throw's gap come
    out positive, scoring it as good sequencing. Back to matching the
    reference exactly; the boundary-artifact risk is instead addressed by
    widening PREROLL_SECONDS/POST_TRIGGER_SECONDS below, which doesn't touch
    this scoring logic at all.)"""
    t = data["timestamps"]

    hip_angle = np.array([
        joint_angle(data["left_hip_xy"][i], data["right_hip_xy"][i], data["right_knee_xy"][i])
        for i in range(len(t))
    ])
    hip_conf = np.minimum(np.minimum(data["left_hip_conf"], data["right_hip_conf"]), data["right_knee_conf"])

    shoulder_angle = np.array([
        joint_angle(data["left_shoulder_xy"][i], data["right_shoulder_xy"][i], data["right_elbow_xy"][i])
        for i in range(len(t))
    ])
    shoulder_conf = np.minimum(np.minimum(data["left_shoulder_conf"], data["right_shoulder_conf"]), data["right_elbow_conf"])

    hip_onset = compute_onset_by_velocity(hip_angle, t, hip_conf)
    shoulder_onset = compute_onset_by_velocity(shoulder_angle, t, shoulder_conf)

    if hip_onset is None or shoulder_onset is None:
        return {"metric": "sequencing", "message": "Not enough confident data to score.", "verdict": None}

    gap = shoulder_onset - hip_onset
    verdict = gap > 0.02
    return {
        "metric": "sequencing",
        "hip_onset": hip_onset,
        "shoulder_onset": shoulder_onset,
        "gap_seconds": gap,
        "verdict": verdict,
        "message": "Great sequencing — hips rotating before shoulders." if verdict
                   else "Throw is becoming arm-dominant — shoulders rotating too early relative to hips.",
    }

def diagnose_sequencing(data, confidence_threshold=0.4):
    """Explains *why* sequencing_score returned verdict=None. Now that
    hip_onset is re-detected again (not taken from the trigger time), either
    side's detection can fail, so both are checked."""
    hip_conf = np.minimum(np.minimum(data["left_hip_conf"], data["right_hip_conf"]), data["right_knee_conf"])
    shoulder_conf = np.minimum(np.minimum(data["left_shoulder_conf"], data["right_shoulder_conf"]), data["right_elbow_conf"])
    n = len(data["timestamps"])
    hip_valid_pct = 100.0 * (hip_conf >= confidence_threshold).sum() / n
    shoulder_valid_pct = 100.0 * (shoulder_conf >= confidence_threshold).sum() / n
    print(f"  [diagnostic] frames in clip: {n}")
    print(f"  [diagnostic] hip-side confidence >= {confidence_threshold}: {hip_valid_pct:.0f}% of frames")
    print(f"  [diagnostic] shoulder-side confidence >= {confidence_threshold}: {shoulder_valid_pct:.0f}% of frames")
    if hip_valid_pct < 30:
        print("  [diagnostic] LIKELY CAUSE: hip/knee tracking confidence too low — check camera angle/distance/lighting on the lower body.")
    if shoulder_valid_pct < 30:
        print("  [diagnostic] LIKELY CAUSE: shoulder/elbow tracking confidence too low — check camera framing on the upper body.")
    if hip_valid_pct >= 30 and shoulder_valid_pct >= 30:
        print("  [diagnostic] confidence looks fine on both sides — the motion in this clip likely never crossed the onset velocity threshold.")
        print("  [diagnostic] consider whether PREROLL_SECONDS/POST_TRIGGER_SECONDS fully bracket the actual throwing motion.")

def draw_skeleton_from_joints(frame_bgr, frame_record, threshold=SKELETON_DRAW_CONF_THRESHOLD):
    positions = {}
    for name in tracked_joints:
        conf = frame_record[f"{name}_conf"]
        if conf < threshold:
            continue
        x, y = frame_record[f"{name}_xy"]
        positions[name] = (int(x), int(y))
        cv2.circle(frame_bgr, positions[name], 5, (0, 255, 0), -1)
    for a, b in LIVE_SKELETON_EDGES:
        if a in positions and b in positions:
            cv2.line(frame_bgr, positions[a], positions[b], (255, 255, 0), 2)

def write_skeleton_video_from_buffer(frames, output_path, fps_estimate):
    """Draws the skeleton straight from joint positions already computed
    during live capture — no second model invocation needed, since we
    already have every joint position from the live inference pass."""
    if not frames:
        return
    h, w = frames[0]["raw"].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, max(fps_estimate, 1.0), (w, h))
    for f in frames:
        overlay_frame = f["raw"].copy()
        draw_skeleton_from_joints(overlay_frame, f)
        writer.write(overlay_frame)
    writer.release()


cap = cv2.VideoCapture(0)

state = "watching"
preroll = deque()  # frames kept while watching, trimmed to PREROLL_SECONDS
capture_buffer = []
capture_trigger_time = None
result_lines = ["Watching for pitch..."]
result_shown_until = 0.0

pitch_count = 0
pitch_history = []  # one entry per completed pitch: {"pitch_number", "sequencing", "skeleton_path", "time"}

velocity_history = deque(maxlen=VELOCITY_BASELINE_SAMPLES)
prev_hip_angle = None
prev_t = None

t0 = time.time()

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        now = time.time() - t0
        display = frame.copy()

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (IN_W, IN_H))
        model_input = np.expand_dims(resized, axis=0).astype(np.uint8)

        interpreter_npu.set_tensor(input_details[0]["index"], model_input)
        interpreter_npu.invoke()
        heatmaps = interpreter_npu.get_tensor(output_details[0]["index"])[0]

        hm_h, hm_w, _ = heatmaps.shape
        img_h, img_w = frame.shape[0], frame.shape[1]

        joint_data = get_frame_joint_data(heatmaps, img_w, img_h, hm_w, hm_h)
        frame_record = {"t": now, "raw": frame, **joint_data}

        hip_angle = joint_angle(joint_data["left_hip_xy"], joint_data["right_hip_xy"], joint_data["right_knee_xy"])
        hip_conf = min(joint_data["left_hip_conf"], joint_data["right_hip_conf"], joint_data["right_knee_conf"])

        # ---------------- state machine ----------------
        if state == "watching":
            preroll.append(frame_record)
            while preroll and now - preroll[0]["t"] > PREROLL_SECONDS:
                preroll.popleft()

            if hip_conf >= HIP_CONF_THRESHOLD and prev_hip_angle is not None:
                velocity = abs(hip_angle - prev_hip_angle) / max(now - prev_t, 1e-6)

                if len(velocity_history) >= 8:
                    baseline_mean = np.mean(velocity_history)
                    baseline_std = np.std(velocity_history) + 1e-6
                    threshold = baseline_mean + VELOCITY_MULTIPLIER * baseline_std

                    if velocity > threshold and velocity > MIN_TRIGGER_VELOCITY:
                        state = "capturing"
                        capture_buffer = list(preroll)  # seed with real "before motion" context
                        capture_trigger_time = now
                        print(f"[{now:.2f}s] Pitch motion detected, capturing...")

                velocity_history.append(velocity)

            if hip_conf >= HIP_CONF_THRESHOLD:
                prev_hip_angle = hip_angle
                prev_t = now

        elif state == "capturing":
            capture_buffer.append(frame_record)
            if now - capture_trigger_time >= POST_TRIGGER_SECONDS:
                span = capture_buffer[-1]["t"] - capture_buffer[0]["t"]
                fps_estimate = len(capture_buffer) / span if span > 0 else 24.0
                clip_data = buffer_to_data_dict(capture_buffer, fps_estimate)

                seq = sequencing_score(clip_data)

                pitch_count += 1

                skel_path = os.path.join(SKEL_OUT_DIR, f"pitch_{pitch_count}_skeleton.mp4")
                write_skeleton_video_from_buffer(capture_buffer, skel_path, fps_estimate)

                pitch_history.append({
                    "pitch_number": pitch_count,
                    "time": now,
                    "sequencing": seq,
                    "skeleton_path": skel_path,
                })

                print(f"=== PITCH #{pitch_count} RESULT ===")
                print("sequencing:", seq)
                print(f"skeleton overlay saved -> {skel_path}")
                if seq.get("verdict") is None:
                    diagnose_sequencing(clip_data)

                result_lines = [
                    f"Pitch #{pitch_count}",
                    f"SEQ: {seq['message']}",
                ]

                state = "showing_results"
                result_shown_until = now + RESULT_DISPLAY_SECONDS
                velocity_history.clear()
                prev_hip_angle = None
                prev_t = None
                preroll.clear()

        elif state == "showing_results":
            if now > result_shown_until:
                state = "watching"

        # ---------------- overlay ----------------
        if state == "watching":
            cv2.putText(display, "Watching for pitch...", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        elif state == "capturing":
            elapsed = now - capture_trigger_time
            cv2.putText(display, f"Capturing pitch... {elapsed:.1f}s", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        elif state == "showing_results":
            for i, line in enumerate(result_lines):
                cv2.putText(display, line, (10, 30 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        # Big, unmistakable live counter, top-right, on every frame regardless of state.
        counter_text = f"PITCHES: {pitch_count}"
        (text_w, text_h), _ = cv2.getTextSize(counter_text, cv2.FONT_HERSHEY_SIMPLEX, 1.3, 3)
        margin = 15
        x = display.shape[1] - text_w - margin
        y = text_h + margin
        cv2.rectangle(display, (x - 10, 0), (display.shape[1], y + margin), (0, 0, 0), -1)
        cv2.putText(display, counter_text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 0), 3)

        cv2.imshow("Live Pitch Detector", display)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()

    print()
    print("=" * 60)
    print(f"SESSION SUMMARY: {pitch_count} pitch(es) detected")
    print("=" * 60)
    seq_true = sum(1 for p in pitch_history if p["sequencing"].get("verdict") is True)
    seq_false = sum(1 for p in pitch_history if p["sequencing"].get("verdict") is False)
    seq_none = pitch_count - seq_true - seq_false
    print(f"sequencing:            {seq_true} good / {seq_false} arm-dominant / {seq_none} unscored")
    print()
    for p in pitch_history:
        print(f"  Pitch #{p['pitch_number']} (t={p['time']:.1f}s):  "
              f"sequencing={p['sequencing'].get('verdict')}  "
              f"skeleton={p['skeleton_path']}")





SESSION SUMMARY: 0 pitch(es) detected
sequencing:            0 good / 0 arm-dominant / 0 unscored
front_side_stability:   0 stable / 0 collapsing / 0 unscored

